# Case Study: Classical and Neural on the Same Task

**Computational Text Analysis Pack · Module 23 — African NLP Models and Benchmarks**

*Technique:* Head-to-head comparison; when the extra machinery is worth it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/13-african-nlp-models/23d_classical_vs_neural.ipynb)

Data: `masakhanews.csv`, `masakhaner.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'masakhaner.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_ner   = pd.read_csv('masakhaner.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

A practice notebook: one research question, worked end to end, with both
toolkits this course has taught.

**The question.** You have a multilingual African news archive and you want to
find every article about health policy, across all sixteen languages. What is
the cheapest method that actually works?

We will answer it with measurements rather than preference.

## The tooling

**HuggingFace `transformers`** is the working layer for every model here: it
downloads the weights, tokenises, and runs the forward pass. Underneath it sits
**PyTorch** (the research default, dynamic and easy to debug); **TensorFlow /
Keras** is the other option, and what the Case Studies module uses.

The models themselves come from African NLP research groups:

- **AfriBERTa** (Ogueji et al.) — trained on 11 African languages, deliberately
  small, and evidence that useful models do not need web-scale data.
- **AfroXLMR** (Alabi et al.) — XLM-R adapted to African languages; the NER model
  in this notebook is its fine-tuned descendant.
- **LaBSE** (Google) — 109 languages, sentence-level, and the reason a Hausa
  sentence and its English translation land in the same place.
- **MasakhaNER / MasakhaNEWS / MasakhaPOS** — the benchmarks, built by the
  Masakhane community. `masakhaner.csv` in this course *is* that benchmark.

### Where the GPU comes from, and why you never need a key

The setup cell picks a backend from wherever you are running:

| Running in | GPU | What you need |
|---|---|---|
| **CourseHub Lab** | a shared L40S, reached through an authenticated CourseHub endpoint | nothing — the credential stays on the server |
| **Colab** | *your own* free GPU (`Runtime > Change runtime type > T4`) | nothing |
| **Anywhere offline** | none; cells use vectors shipped with the course | nothing |

There is no API key in this course, by design. A notebook cell is readable by
whoever runs it, so a key pasted into one is a key given away.

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
BACKEND = "cached"
_pod = _hf = None

if IN_COLAB:
    # Colab hands every participant their own GPU. Use it rather than sharing
    # one server: it is faster for a room full of people, and needs no key.
    try:
        import transformers  # noqa: F401
    except ImportError:
        os.system("pip install -q transformers")
    import torch
    BACKEND = "colab-gpu" if torch.cuda.is_available() else "colab-cpu"
else:
    try:
        sys.path.insert(0, "/app")      # the CourseHub runner executes inside /app
        from utils import gpu_pod as _pod
        _info = _pod.health()
        BACKEND = "coursehub-pod"
    except Exception as exc:
        _why = str(exc)[:80]

POD_LIVE = BACKEND in ("coursehub-pod", "colab-gpu", "colab-cpu")

def _hf_pipe(kind, repo):
    global _hf
    _hf = _hf or {}
    if (kind, repo) not in _hf:
        from transformers import AutoModel, AutoTokenizer, pipeline
        import torch
        dev = 0 if torch.cuda.is_available() else -1
        if kind == "ner":
            _hf[(kind, repo)] = pipeline("token-classification", model=repo,
                                         aggregation_strategy="simple", device=dev)
        else:
            _hf[(kind, repo)] = (AutoTokenizer.from_pretrained(repo),
                                 AutoModel.from_pretrained(repo))
    return _hf[(kind, repo)]

def embed(texts, model="labse"):
    """Sentence embeddings, wherever we are. Returns a numpy array."""
    import numpy as np
    if BACKEND == "coursehub-pod":
        return np.asarray(_pod.embed(list(texts), model=model)["vectors"], dtype="float32")
    if BACKEND.startswith("colab"):
        import torch
        tok, mod = _hf_pipe("embed", "sentence-transformers/LaBSE")
        if torch.cuda.is_available():
            mod = mod.cuda()
        with torch.no_grad():
            b = tok(list(texts), padding=True, truncation=True,
                    max_length=128, return_tensors="pt")
            b = {k: v.to(mod.device) for k, v in b.items()}
            out = mod(**b).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).float()
            v = (out * m).sum(1) / m.sum(1).clamp(min=1e-9)
            v = torch.nn.functional.normalize(v, dim=-1)
        return v.cpu().numpy().astype("float32")
    raise RuntimeError("no GPU backend available; use the shipped vectors instead")

def ner(texts, model="masakhaner"):
    """Named entities from the MasakhaNER 2.0 benchmark model."""
    repo = "masakhane/afroxlmr-large-ner-masakhaner-1.0_2.0"
    if BACKEND == "coursehub-pod":
        return _pod.ner(list(texts), model=model)["results"]
    if BACKEND.startswith("colab"):
        pipe = _hf_pipe("ner", repo)
        out = pipe(list(texts))
        if out and isinstance(out[0], dict):
            out = [out]
        return [[{"text": e["word"], "label": e["entity_group"],
                  "score": round(float(e["score"]), 4)} for e in ents] for ents in out]
    raise RuntimeError("no GPU backend available")

print(f"backend: {BACKEND}")
if BACKEND == "coursehub-pod":
    print(f"   {_info['gpu']}, {_info['vram_total_mb']:,} MB VRAM (shared, via CourseHub)")
elif BACKEND == "colab-gpu":
    import torch; print(f"   {torch.cuda.get_device_name(0)} (your own Colab GPU)")
elif BACKEND == "colab-cpu":
    print("   Colab without a GPU — Runtime > Change runtime type > T4 GPU makes this faster")
else:
    print(f"   no live backend ({_why}); cells fall back to the vectors shipped with the course")

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

## Contender 1 — TF-IDF and logistic regression (module 7)

No pretraining, no GPU, runs in seconds on a laptop.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import time

sub = df[df["n_words"] >= 30].reset_index(drop=True)
Xtr, Xte, ytr, yte = train_test_split(sub["text"], sub["category"], test_size=0.25,
                                      random_state=0, stratify=sub["category"])
t0 = time.time()
vec = TfidfVectorizer(token_pattern=r"\w+", min_df=3, max_features=20000,
                      sublinear_tf=True)
A, B = vec.fit_transform(Xtr), vec.transform(Xte)
clf = LogisticRegression(max_iter=1000).fit(A, ytr)
pred_lr = clf.predict(B)
lr_time = time.time() - t0
print(f"TF-IDF + logistic regression")
print(f"   accuracy {accuracy_score(yte, pred_lr):.3f}   "
      f"macro-F1 {f1_score(yte, pred_lr, average='macro'):.3f}   "
      f"{lr_time:.1f}s on CPU")

## Contender 2 — LaBSE embeddings plus the same classifier

Swap the features. The classifier is identical; only the representation changes
from word counts to pretrained multilingual vectors.

In [ ]:
index = pd.read_csv("labse_headlines_index.csv")
bank = np.load("labse_headlines.npy")
print(f"shipped vectors: {bank.shape[0]} headlines x {bank.shape[1]} dims, "
      f"{index['language'].nunique()} languages")

mask = index["category"].notna()
Etr, Ete, etr, ete = train_test_split(bank[mask.values], index.loc[mask, "category"],
                                      test_size=0.3, random_state=0,
                                      stratify=index.loc[mask, "category"])
t0 = time.time()
emb_clf = LogisticRegression(max_iter=2000).fit(Etr, etr)
pred_emb = emb_clf.predict(Ete)
emb_time = time.time() - t0
print(f"\nLaBSE embeddings + logistic regression")
print(f"   accuracy {accuracy_score(ete, pred_emb):.3f}   "
      f"macro-F1 {f1_score(ete, pred_emb, average='macro'):.3f}   "
      f"{emb_time:.1f}s (vectors precomputed on GPU)")
print()
print("Careful: these two numbers are NOT directly comparable. The first uses")
print("full article text, the second uses headlines only, because that is what")
print("was precomputed. Read them as two workflows, not as a league table.")

## The comparison that is fair

To compare properly, both methods must see the same text. Here both get the
headlines only.

In [ ]:
head = index.loc[mask].reset_index(drop=True)
vecs = bank[mask.values]
tr_i, te_i = train_test_split(np.arange(len(head)), test_size=0.3,
                              random_state=0, stratify=head["category"])

v2 = TfidfVectorizer(token_pattern=r"\w+", min_df=2, max_features=20000)
A2 = v2.fit_transform(head["headline"].iloc[tr_i].astype(str))
B2 = v2.transform(head["headline"].iloc[te_i].astype(str))
lr2 = LogisticRegression(max_iter=1000).fit(A2, head["category"].iloc[tr_i])
p_tfidf = lr2.predict(B2)

lr3 = LogisticRegression(max_iter=2000).fit(vecs[tr_i], head["category"].iloc[tr_i])
p_lab = lr3.predict(vecs[te_i])

gold = head["category"].iloc[te_i]
comp = pd.DataFrame({
    "method": ["TF-IDF (headlines)", "LaBSE (headlines)"],
    "accuracy": [accuracy_score(gold, p_tfidf), accuracy_score(gold, p_lab)],
    "macro_F1": [f1_score(gold, p_tfidf, average="macro"),
                 f1_score(gold, p_lab, average="macro")],
})
print(comp.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
# Where the embeddings earn their keep: the smaller languages.
per_lang = []
langs_te = head["language"].iloc[te_i].values
for lang in sorted(set(langs_te)):
    m = langs_te == lang
    if m.sum() < 5:
        continue
    per_lang.append({"language": lang, "n": int(m.sum()),
                     "TFIDF": accuracy_score(gold[m], p_tfidf[m]),
                     "LaBSE": accuracy_score(gold[m], p_lab[m])})
pl = pd.DataFrame(per_lang).sort_values("n")
print(pl.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

fig, ax = plt.subplots(figsize=(8, 4.5))
y = np.arange(len(pl))
ax.barh(y - .2, pl["TFIDF"], height=.4, label="TF-IDF")
ax.barh(y + .2, pl["LaBSE"], height=.4, label="LaBSE")
ax.set_yticks(y); ax.set_yticklabels(pl["language"], fontsize=8)
ax.set_xlabel("accuracy"); ax.legend(); ax.set_title("Per language, same text")
plt.tight_layout()

## How to decide, for your own project

Neither answer is "always". The decision rule that survives contact with a real
project:

1. **Start classical.** It is faster, interpretable, needs no GPU, and on a
   single language with plenty of labelled text it is often as good.
2. **Reach for pretrained embeddings when you are crossing languages**, or when
   labelled data is scarce — that is where transfer pays.
3. **Check per language, never on the average.** An overall score hides that the
   model is failing on exactly the languages your project may care most about.
4. **Report which model, which version, which date.** These models are updated;
   a result from `afroxlmr-large-ner-masakhaner-1.0_2.0` is not reproducible
   from "we used AfroXLMR".

### 🔧 Your turn 1

Take a research question of your own, and decide which of the two workflows fits it. Write down the reason in one sentence, then test it here.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

The commonest honest outcome is that the classical method is adequate and the
neural one is not worth the dependency — which is a real finding, not a
concession. The cases where the neural workflow clearly wins share a shape:
several languages at once, or very little labelled data, or a need for semantic
rather than lexical matching. If your question has none of those properties, the
first 22 modules of this course already contain your answer, and it will be
easier to defend in a footnote.

</details>